# CLS No-Groups Sensitivity Check

This notebook runs a TEST-only sensitivity analysis for the fixed final CLS baseline without the raw group features `group12` and `group34`.

Locked scope:
- train data: `cls_train_full_no_groups.csv`
- test data: `cls_test_no_groups.csv`
- model: `HistGradientBoosting` only
- no `LightGBM`, `CatBoost`, `Logistic Regression`, or other challenger models
- no `cls_val.csv` and no `cls_val_no_groups.csv`
- no new final model decision; the final CLS baseline remains the base HistGradientBoosting model at threshold `0.22`

## Run Policy

This is a documentation-only sensitivity analysis.

Operational rules:
- full-train is allowed for this run because the no-groups variant uses fewer features and only one HistGradientBoosting model
- threshold tuning uses the same grid as the base model: `0.05` to `0.80` in `0.01` steps
- if the full no-groups HistGradientBoosting fit exceeds 45 minutes, stop and document a technical limitation
- if that technical limitation is hit, do not run VAL and do not auto-start any fallback run

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 20)

print("Setup complete. HistGradientBoosting-only sensitivity notebook ready.")

Setup complete. HistGradientBoosting-only sensitivity notebook ready.


In [2]:
BASE_DIR = Path(r"c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS")
DATA_DIR = BASE_DIR / "Daten"
TRAIN_PATH = DATA_DIR / "cls_train_full_no_groups.csv"
TEST_PATH = DATA_DIR / "cls_test_no_groups.csv"

DEFAULT_METRICS_PATH = BASE_DIR / "cls_no_groups_test_metrics_default.csv"
THRESHOLD_RESULTS_PATH = BASE_DIR / "cls_no_groups_threshold_tuning_results.csv"
BEST_MODEL_PATH = BASE_DIR / "cls_no_groups_best_model.csv"
PREDICTIONS_PATH = BASE_DIR / "cls_no_groups_predictions.csv"
CONFUSION_MATRIX_PATH = BASE_DIR / "cls_no_groups_confusion_matrix.csv"

TARGET_COLUMN = "order"
RANDOM_STATE = 42
THRESHOLDS = np.round(np.arange(5, 81) / 100, 2)
EXPECTED_THRESHOLD_COUNT = 76
FIT_TIME_LIMIT_SECONDS = 45 * 60

LEAKAGE_COLUMNS = [
    "lineID",
    "revenue",
    "quantity",
    "q_raw",
    "quantity_class",
    "qty_suspicious",
    "click",
    "basket",
]

CATEGORICAL_CANDIDATES = [
    "salesIndex",
    "category_norm",
    "pharmForm_norm",
    "has_campaign",
    "pid_segment",
    "group12",
    "group34",
    "price_diff_bin",
    "discount_bin",
]

BASE_TEST_REFERENCE = {
    "model": "HistGradientBoosting",
    "threshold": 0.22,
    "auc": 0.719114,
    "logloss": 0.493439,
    "f1": 0.482480,
    "precision": 0.360131,
    "recall": 0.730735,
    "mcc": 0.274325,
    "tn": 156832,
    "fp": 108809,
    "fn": 22566,
    "tp": 61240,
}

HISTGB_PARAMS = {
    "learning_rate": 0.05,
    "max_iter": 300,
    "max_depth": 8,
    "random_state": RANDOM_STATE,
}

LOADED_DATASET_PATHS = []

assert len(THRESHOLDS) == EXPECTED_THRESHOLD_COUNT

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Threshold count: {len(THRESHOLDS)}")
print(f"HistGB params: {HISTGB_PARAMS}")
print(f"Fit time limit (seconds): {FIT_TIME_LIMIT_SECONDS}")

Train path: c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS\Daten\cls_train_full_no_groups.csv
Test path: c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS\Daten\cls_test_no_groups.csv
Threshold count: 76
HistGB params: {'learning_rate': 0.05, 'max_iter': 300, 'max_depth': 8, 'random_state': 42}
Fit time limit (seconds): 2700


In [3]:
def print_target_distribution(split_name: str, series: pd.Series) -> None:
    counts = series.value_counts(dropna=False).sort_index()
    order_rate = pd.to_numeric(series, errors="raise").mean() * 100
    print(f"{split_name} target distribution:")
    print(counts.to_string())
    print(f"{split_name} order rate (%): {order_rate:.4f}")


train_df = pd.read_csv(TRAIN_PATH)
LOADED_DATASET_PATHS.append(str(TRAIN_PATH))
test_df = pd.read_csv(TEST_PATH)
LOADED_DATASET_PATHS.append(str(TEST_PATH))

if TARGET_COLUMN not in train_df.columns or TARGET_COLUMN not in test_df.columns:
    raise KeyError(f"Target column '{TARGET_COLUMN}' must exist in both train and test")

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print()
print_target_distribution("Train", train_df[TARGET_COLUMN])
print()
print_target_distribution("Test", test_df[TARGET_COLUMN])

Train shape: (1521260, 27)
Test shape: (349447, 27)

Train target distribution:
order
0    1182841
1     338419
Train order rate (%): 22.2460

Test target distribution:
order
0    265641
1     83806
Test order rate (%): 23.9825


In [4]:
y_train = pd.to_numeric(train_df[TARGET_COLUMN], errors="raise").astype(int)
y_test = pd.to_numeric(test_df[TARGET_COLUMN], errors="raise").astype(int)

for split_name, target_series in [("Train", y_train), ("Test", y_test)]:
    unique_values = set(target_series.unique().tolist())
    if unique_values != {0, 1}:
        raise ValueError(f"{split_name} target must contain only 0/1, found {sorted(unique_values)}")

raw_train_feature_columns = [column for column in train_df.columns if column != TARGET_COLUMN]
raw_test_feature_columns = [column for column in test_df.columns if column != TARGET_COLUMN]

if raw_train_feature_columns != raw_test_feature_columns:
    missing_in_test = [column for column in raw_train_feature_columns if column not in raw_test_feature_columns]
    additional_in_test = [column for column in raw_test_feature_columns if column not in raw_train_feature_columns]
    raise ValueError(
        "Train/Test feature mismatch detected. "
        f"Missing in test: {missing_in_test}. Additional in test: {additional_in_test}."
    )

if "group12" in raw_train_feature_columns or "group34" in raw_train_feature_columns:
    raise ValueError("The no-groups train export must not contain group12 or group34.")
if "group12" in raw_test_feature_columns or "group34" in raw_test_feature_columns:
    raise ValueError("The no-groups test export must not contain group12 or group34.")

present_leakage_columns = [column for column in LEAKAGE_COLUMNS if column in raw_train_feature_columns]
removed_leakage_columns = present_leakage_columns.copy()
already_absent_leakage_columns = [column for column in LEAKAGE_COLUMNS if column not in raw_train_feature_columns]

feature_columns = [column for column in raw_train_feature_columns if column not in LEAKAGE_COLUMNS]
X_train_raw = train_df[feature_columns].copy()
X_test_raw = test_df[feature_columns].copy()

if any(column in X_train_raw.columns for column in LEAKAGE_COLUMNS):
    raise ValueError("Leakage columns are still present in X_train_raw")
if any(column in X_test_raw.columns for column in LEAKAGE_COLUMNS):
    raise ValueError("Leakage columns are still present in X_test_raw")

group_aggregate_columns_present = [column for column in ["group12_order", "group34_order"] if column in feature_columns]

print("Leakage columns present in exports:")
print(present_leakage_columns if present_leakage_columns else "None")
print("\nLeakage columns removed from modeling features:")
print(removed_leakage_columns if removed_leakage_columns else "None")
print("\nLeakage columns already absent:")
print(already_absent_leakage_columns if already_absent_leakage_columns else "None")
print(f"\nFeature count after leakage removal: {len(feature_columns)}")
print("Train/Test feature columns match exactly after target removal and leakage filtering.")
print(f"Group aggregate features retained by agreed scope: {group_aggregate_columns_present}")

Leakage columns present in exports:
None

Leakage columns removed from modeling features:
None

Leakage columns already absent:
['lineID', 'revenue', 'quantity', 'q_raw', 'quantity_class', 'qty_suspicious', 'click', 'basket']

Feature count after leakage removal: 26
Train/Test feature columns match exactly after target removal and leakage filtering.
Group aggregate features retained by agreed scope: ['group12_order', 'group34_order']


In [5]:
categorical_features = [
    column for column in CATEGORICAL_CANDIDATES
    if column in feature_columns and column != "availability"
]
numeric_features = [column for column in feature_columns if column not in categorical_features]

if "availability" in categorical_features:
    raise ValueError("availability must remain numeric/ordinal, not categorical")
if "availability" in feature_columns and "availability" not in numeric_features:
    raise ValueError("availability should be part of the numeric feature set")

print(f"Categorical feature count: {len(categorical_features)}")
print(categorical_features)
print(f"\nNumeric feature count: {len(numeric_features)}")
print(numeric_features)

Categorical feature count: 7
['salesIndex', 'category_norm', 'pharmForm_norm', 'has_campaign', 'pid_segment', 'price_diff_bin', 'discount_bin']

Numeric feature count: 19
['day', 'day_7', 'day_14', 'day_30', 'adFlag', 'availability', 'price', 'competitorPrice', 'is_greater_discount', 'price_per_unit', 'pid_total_events', 'click_time', 'basket_time', 'order_time', 'group12_order', 'group34_order', 'pid_prob', 'availability_likelihood', 'day_7_likelihood']


In [6]:
def safe_logloss(y_true: pd.Series, y_prob: np.ndarray) -> float:
    clipped = np.clip(np.asarray(y_prob, dtype=float), 1e-15, 1 - 1e-15)
    return float(log_loss(y_true, clipped, labels=[0, 1]))


def compute_threshold_metrics(
    y_true: pd.Series,
    y_prob: np.ndarray,
    threshold: float,
    model_name: str,
    auc_value: float,
    logloss_value: float,
) -> dict:
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "threshold": float(threshold),
        "auc": float(auc_value),
        "logloss": float(logloss_value),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def evaluate_threshold_grid(
    y_true: pd.Series,
    y_prob: np.ndarray,
    model_name: str,
    auc_value: float,
    logloss_value: float,
    thresholds: np.ndarray,
) -> pd.DataFrame:
    rows = [
        compute_threshold_metrics(y_true, y_prob, threshold, model_name, auc_value, logloss_value)
        for threshold in thresholds
    ]
    return pd.DataFrame(rows)


def rank_by_f1(results: pd.DataFrame) -> pd.DataFrame:
    ranked = results.assign(threshold_distance=(results["threshold"] - 0.50).abs())
    ranked = ranked.sort_values(
        by=["f1", "mcc", "precision", "recall", "threshold_distance", "model"],
        ascending=[False, False, False, False, True, True],
    )
    return ranked.drop(columns=["threshold_distance"]).reset_index(drop=True)


def rank_by_mcc(results: pd.DataFrame) -> pd.DataFrame:
    ranked = results.assign(threshold_distance=(results["threshold"] - 0.50).abs())
    ranked = ranked.sort_values(
        by=["mcc", "f1", "precision", "recall", "threshold_distance", "model"],
        ascending=[False, False, False, False, True, True],
    )
    return ranked.drop(columns=["threshold_distance"]).reset_index(drop=True)


def select_best_threshold(model_results: pd.DataFrame, default_row: dict) -> tuple[dict, dict]:
    ranked = rank_by_f1(model_results)
    top_candidate = ranked.iloc[0].to_dict()
    default_precision = float(default_row["precision"])
    default_recall = float(default_row["recall"])
    precision_ratio = np.nan
    precision_collapse = False
    precision_message = "No precision collapse detected relative to the 0.50 baseline."

    if default_precision > 0:
        precision_ratio = float(top_candidate["precision"]) / default_precision
        precision_collapse = (
            float(top_candidate["precision"]) < 0.5 * default_precision
            and float(top_candidate["recall"]) > default_recall
        )

    if precision_collapse:
        plausible_mask = ~(
            (ranked["precision"] < 0.5 * default_precision)
            & (ranked["recall"] > default_recall)
        )
        plausible_candidates = ranked.loc[plausible_mask].reset_index(drop=True)
        if not plausible_candidates.empty:
            top_candidate = plausible_candidates.iloc[0].to_dict()
            precision_message = (
                "Top F1 candidate triggered the precision-collapse guard relative to 0.50; the best plausible alternative was selected."
            )
        else:
            precision_message = (
                "Top F1 candidate triggered the precision-collapse guard, but no alternative candidate satisfied the plausibility filter."
            )

    precision_check = {
        "precision_collapse_vs_050": bool(precision_collapse),
        "precision_ratio_vs_050": precision_ratio,
        "precision_check_message": precision_message,
    }
    return top_candidate, precision_check


def prepare_histgb_frames(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    categorical_cols: list[str],
    numeric_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_prepared = pd.DataFrame(index=train_frame.index)
    test_prepared = pd.DataFrame(index=test_frame.index)

    for column in numeric_cols:
        train_prepared[column] = pd.to_numeric(train_frame[column], errors="coerce")
        test_prepared[column] = pd.to_numeric(test_frame[column], errors="coerce")

    for column in categorical_cols:
        train_series = train_frame[column].astype("string").fillna("__MISSING__")
        test_series = test_frame[column].astype("string").fillna("__MISSING__")
        categories = pd.Index(pd.unique(train_series))
        mapping = {category: index for index, category in enumerate(categories)}
        train_prepared[column] = train_series.map(mapping).astype(float)
        test_prepared[column] = test_series.map(mapping).fillna(-1).astype(float)

    return train_prepared[feature_columns], test_prepared[feature_columns]

In [7]:
run_start = time.perf_counter()
X_train_histgb, X_test_histgb = prepare_histgb_frames(
    X_train_raw,
    X_test_raw,
    categorical_features,
    numeric_features,
)

estimator = HistGradientBoostingClassifier(**HISTGB_PARAMS)
fit_start = time.perf_counter()
estimator.fit(X_train_histgb, y_train)
fit_seconds = time.perf_counter() - fit_start
run_seconds = time.perf_counter() - run_start

if fit_seconds > FIT_TIME_LIMIT_SECONDS:
    raise TimeoutError(
        "The full no-groups HistGradientBoosting run exceeded the 45-minute limit. Document this as a technical limitation and do not run any VAL workflow. Optional next step: prepare a 300k/full-test sensitivity check, but do not auto-start it."
    )

y_prob = estimator.predict_proba(X_test_histgb)[:, 1]
auc_value = float(roc_auc_score(y_test, y_prob))
logloss_value = safe_logloss(y_test, y_prob)

default_row = compute_threshold_metrics(
    y_true=y_test,
    y_prob=y_prob,
    threshold=0.50,
    model_name=BASE_TEST_REFERENCE["model"],
    auc_value=auc_value,
    logloss_value=logloss_value,
)
default_row["backend"] = "hist_gradient_boosting"
default_row["fit_seconds"] = fit_seconds
default_row["run_seconds"] = run_seconds
default_metrics_df = pd.DataFrame([default_row])

threshold_results_df = evaluate_threshold_grid(
    y_true=y_test,
    y_prob=y_prob,
    model_name=BASE_TEST_REFERENCE["model"],
    auc_value=auc_value,
    logloss_value=logloss_value,
    thresholds=THRESHOLDS,
)
threshold_results_df["backend"] = "hist_gradient_boosting"
threshold_results_df["fit_seconds"] = fit_seconds
threshold_results_df["run_seconds"] = run_seconds

best_row, precision_check = select_best_threshold(threshold_results_df, default_row)
best_row["backend"] = "hist_gradient_boosting"
best_row["fit_seconds"] = fit_seconds
best_row["run_seconds"] = run_seconds
best_row.update(precision_check)
best_row["selection_method"] = "F1 primary, MCC tie-breaker"
best_model_df = pd.DataFrame([best_row])

predictions_df = pd.DataFrame(
    {
        "model": BASE_TEST_REFERENCE["model"],
        "backend": "hist_gradient_boosting",
        "row_index": test_df.index.to_numpy(),
        TARGET_COLUMN: y_test.to_numpy(),
        "p_order_1": y_prob,
        "pred_default_050": (y_prob >= 0.50).astype(int),
        "best_threshold": float(best_row["threshold"]),
        "pred_best_threshold": (y_prob >= float(best_row["threshold"])).astype(int),
    }
)

confusion_matrix_df = pd.DataFrame([
    {
        "model": BASE_TEST_REFERENCE["model"],
        "backend": "hist_gradient_boosting",
        "threshold": float(best_row["threshold"]),
        "tn": int(best_row["tn"]),
        "fp": int(best_row["fp"]),
        "fn": int(best_row["fn"]),
        "tp": int(best_row["tp"]),
    }
])

delta_row = {
    "delta_auc_vs_base_test": float(best_row["auc"]) - BASE_TEST_REFERENCE["auc"],
    "delta_logloss_vs_base_test": float(best_row["logloss"]) - BASE_TEST_REFERENCE["logloss"],
    "delta_f1_vs_base_test": float(best_row["f1"]) - BASE_TEST_REFERENCE["f1"],
    "delta_precision_vs_base_test": float(best_row["precision"]) - BASE_TEST_REFERENCE["precision"],
    "delta_recall_vs_base_test": float(best_row["recall"]) - BASE_TEST_REFERENCE["recall"],
    "delta_mcc_vs_base_test": float(best_row["mcc"]) - BASE_TEST_REFERENCE["mcc"],
}
delta_df = pd.DataFrame([delta_row])

significantly_worse = (
    delta_row["delta_auc_vs_base_test"] < -0.01
    or delta_row["delta_f1_vs_base_test"] < -0.02
    or delta_row["delta_mcc_vs_base_test"] < -0.02
    or delta_row["delta_logloss_vs_base_test"] > 0.02
)
significantly_better = (
    delta_row["delta_auc_vs_base_test"] > 0.005
    or delta_row["delta_f1_vs_base_test"] > 0.01
    or delta_row["delta_mcc_vs_base_test"] > 0.01
)

if significantly_worse:
    interpretation = "No-groups is clearly worse than the base TEST reference. group12/group34 likely provide genuine signal gain in the final CLS setup."
elif significantly_better:
    interpretation = "No-groups is better than the base TEST reference on TEST. Document this as a limitation, but do not reopen the final decision because VAL was already used for the fixed base model."
else:
    interpretation = "No-groups is close to the base TEST reference. The final CLS model appears robust to removing the raw group features group12/group34."

DEFAULT_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
default_metrics_df.to_csv(DEFAULT_METRICS_PATH, index=False)
threshold_results_df.to_csv(THRESHOLD_RESULTS_PATH, index=False)
best_model_df.to_csv(BEST_MODEL_PATH, index=False)
predictions_df.to_csv(PREDICTIONS_PATH, index=False)
confusion_matrix_df.to_csv(CONFUSION_MATRIX_PATH, index=False)

top10_f1_df = rank_by_f1(threshold_results_df).head(10).reset_index(drop=True)
top10_mcc_df = rank_by_mcc(threshold_results_df).head(10).reset_index(drop=True)

print(f"HistGB full no-groups fit completed in {fit_seconds:.2f} seconds.")
print(f"Total run time: {run_seconds:.2f} seconds.")
print("Artifacts written.")

HistGB full no-groups fit completed in 32.49 seconds.
Total run time: 34.94 seconds.
Artifacts written.


In [8]:
print("Default-threshold metrics (0.50):")
print(default_metrics_df.to_string(index=False))

print("\nTop 10 by F1:")
print(top10_f1_df.to_string(index=False))

print("\nTop 10 by MCC:")
print(top10_mcc_df.to_string(index=False))

print("\nBest no-groups threshold row:")
print(best_model_df.to_string(index=False))

print("\nBest-threshold confusion matrix:")
print(confusion_matrix_df.to_string(index=False))

print("\nDeltas versus base TEST:")
print(delta_df.to_string(index=False))

print("\nDocumentation-ready interpretation:")
print(interpretation)

print("\nRequired close-out:")
print(f"- Laufzeit (fit seconds): {fit_seconds:.2f}")
print(f"- Laufzeit (total seconds): {run_seconds:.2f}")
print(f"- Bester Threshold: {float(best_row['threshold']):.2f}")
print(f"- AUC: {float(best_row['auc']):.6f}")
print(f"- LogLoss: {float(best_row['logloss']):.6f}")
print(f"- F1: {float(best_row['f1']):.6f}")
print(f"- Precision: {float(best_row['precision']):.6f}")
print(f"- Recall: {float(best_row['recall']):.6f}")
print(f"- MCC: {float(best_row['mcc']):.6f}")
print(f"- Confusion Matrix: TN={int(best_row['tn'])}, FP={int(best_row['fp'])}, FN={int(best_row['fn'])}, TP={int(best_row['tp'])}")

Default-threshold metrics (0.50):
               model  threshold      auc  logloss  accuracy  precision   recall       f1      mcc     tn   fp    fn   tp                backend  fit_seconds  run_seconds
HistGradientBoosting        0.5 0.718933 0.493461  0.763212   0.539422 0.086617 0.149265 0.140425 259443 6198 76547 7259 hist_gradient_boosting    32.487421    34.939936

Top 10 by F1:
               model  threshold      auc  logloss  accuracy  precision   recall       f1      mcc     tn     fp    fn    tp                backend  fit_seconds  run_seconds
HistGradientBoosting       0.22 0.718933 0.493461  0.622850   0.359540 0.732859 0.482410 0.274189 156235 109406 22388 61418 hist_gradient_boosting    32.487421    34.939936
HistGradientBoosting       0.23 0.718933 0.493461  0.633613   0.364621 0.710677 0.481964 0.273826 161855 103786 24247 59559 hist_gradient_boosting    32.487421    34.939936
HistGradientBoosting       0.24 0.718933 0.493461  0.643909   0.369960 0.689616 0.481571 0.2

In [9]:
assert train_df is not None and test_df is not None
assert TARGET_COLUMN in train_df.columns and TARGET_COLUMN in test_df.columns
assert set(y_train.unique().tolist()) == {0, 1}
assert set(y_test.unique().tolist()) == {0, 1}
assert raw_train_feature_columns == raw_test_feature_columns
assert "group12" not in raw_train_feature_columns and "group34" not in raw_train_feature_columns
assert "group12" not in raw_test_feature_columns and "group34" not in raw_test_feature_columns
assert all(column not in X_train_raw.columns for column in LEAKAGE_COLUMNS)
assert all(column not in X_test_raw.columns for column in LEAKAGE_COLUMNS)
assert all("val" not in Path(path).name.lower() for path in LOADED_DATASET_PATHS), LOADED_DATASET_PATHS
threshold_counts = threshold_results_df.groupby("model")["threshold"].nunique()
assert (threshold_counts == EXPECTED_THRESHOLD_COUNT).all(), threshold_counts.to_dict()
for required_output in [
    DEFAULT_METRICS_PATH,
    THRESHOLD_RESULTS_PATH,
    BEST_MODEL_PATH,
    PREDICTIONS_PATH,
    CONFUSION_MATRIX_PATH,
]:
    assert required_output.exists(), f"Missing required output file: {required_output}"

print("All no-groups sensitivity assertions passed.")

All no-groups sensitivity assertions passed.
